# Does any single FFN feature matter in a deployed forecasting foundation model?

**What this reproduces.** The Titans experiments in this repo ask where a *test-time*
memory stores things, and find no single hidden unit owns any stored association. This
notebook asks the same question of a **frozen, deployed forecaster** -- Amazon's
Chronos-Bolt -- swapping recall-cosine for a real forecasting metric.

### Why ablation, and not a logit lens

Chronos-Bolt is a T5 encoder-decoder that chunks the context into patches and **regresses
9 quantiles directly**. Its `vocab_size` is 2 -- there is no token vocabulary, so
weight-space lenses have nothing to read into. Ablation needs no vocabulary: zero one FFN
feature, re-forecast, measure the change in Weighted Quantile Loss (WQL).

### The experiment, in order

| # | Step | Why it is there |
|---|---|---|
| 1 | Baseline WQL vs a naive last-value forecast | is the model actually forecasting? |
| 2 | **Positive control**: zero ALL FFN features | if this does not break, ablation is a no-op and nothing below means anything |
| 3 | Ablate each feature individually | does any one feature carry accuracy? |
| 4 | Ablate k random features, sweep k | how redundant is the FFN? |
| 5 | Repeat across datasets | does it generalise? |

Step 2 is the one people skip. Run it first.

Runtime: ~15 min on CPU for everything, ~14 of which is the exhaustive sweep.

## 1 - Setup

In [ ]:
# transformers >=4.48 breaks chronos on some Python versions; pin below it.
!pip install -q "transformers>=4.40,<4.48" chronos-forecasting certifi
import torch, numpy as np, os, shutil, ssl, urllib.request, time, json
print("torch", torch.__version__)

In [ ]:
from chronos import BaseChronosPipeline

MODEL   = "amazon/chronos-bolt-tiny"   # 9M. -base (205M) is the one to quote in a paper
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
LEVELS  = np.arange(0.1, 1.0, 0.1)     # the 9 quantiles Chronos-Bolt emits
CONTEXT, HORIZON = 512, 64

pipe = BaseChronosPipeline.from_pretrained(MODEL, device_map=DEVICE, torch_dtype=torch.float32)
model = pipe.model.eval()
for p in model.parameters():
    p.requires_grad_(False)          # we mutate weights in place; no autograd wanted

mods = dict(model.named_modules())
# encoder FFN lives at layer.1, decoder FFN at layer.2 in T5
FFN = [mods[f"{s}.block.{b}.layer.{1 if s=='encoder' else 2}.DenseReluDense"]
       for s in ("encoder", "decoder") for b in range(4)]
NAMES  = [f"{s}{b}" for s in ("encoder", "decoder") for b in range(4)]
N_FEAT = FFN[0].wi.weight.shape[0]
TOTAL  = len(FFN) * N_FEAT
print(f"{MODEL} on {DEVICE}: {len(FFN)} FFN layers x {N_FEAT} features = {TOTAL}")
print("wi (reads):", tuple(FFN[0].wi.weight.shape), " wo (writes):", tuple(FFN[0].wo.weight.shape))

## 2 - Data and metric

ETT is a standard forecasting benchmark (electricity transformer sensors, 7 channels).
We build non-overlapping windows: 512 observations of context, 64 to predict.

**WQL** (Weighted Quantile Loss) is the metric Chronos reports. For quantile level $q$ the
pinball loss is $\max(q\cdot d,\ (q-1)\cdot d)$ with $d = y - \hat{y}_q$; WQL sums that
over quantiles and timesteps, normalised by $\sum|y|$. **Lower is better.**

In [ ]:
ETT = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/"

def download(url, path):
    if os.path.exists(path): return path
    try:
        import certifi; ctx = ssl.create_default_context(cafile=certifi.where())
    except Exception:
        ctx = None
    with urllib.request.urlopen(urllib.request.Request(url), context=ctx) as r, open(path,"wb") as f:
        shutil.copyfileobj(r, f)
    return path

def load_ett(name):
    return np.genfromtxt(download(ETT+name+".csv", name+".csv"),
                         delimiter=",", skip_header=1, usecols=range(1,8))

def make_windows(arr, context=CONTEXT, horizon=HORIZON, n_win=8):
    ins, tgt = [], []
    for w in range(n_win):
        s = w * context
        if s + context + horizon > len(arr): break
        for ch in range(arr.shape[1]):
            ins.append(torch.tensor(arr[s:s+context, ch], dtype=torch.float32))
            tgt.append(arr[s+context:s+context+horizon, ch])
    return ins, np.stack(tgt)

def wql(qf, y):
    d = y[:, :, None] - qf
    return float(2*np.maximum(LEVELS*d, (LEVELS-1)*d).sum() / np.abs(y).sum())

@torch.no_grad()
def score(ins, tgt, horizon=HORIZON):
    q, _ = pipe.predict_quantiles(inputs=ins, prediction_length=horizon,
                                  quantile_levels=list(LEVELS))
    return wql(q.detach().cpu().numpy(), tgt)

ins, tgt = make_windows(load_ett("ETTh1"))
naive = np.stack([np.full((HORIZON, len(LEVELS)), x[-1].item()) for x in ins])
base  = score(ins, tgt)
print(f"eval set          : {len(ins)} windows (8 x 7 channels)")
print(f"naive last-value  : WQL {wql(naive, tgt):.4f}")
print(f"{MODEL.split('/')[-1]:<18}: WQL {base:.4f}")
print("\n-> the model must clearly beat naive, or ablation effects are meaningless")

## 3 - Positive control (run this before anything else)

Zeroing every FFN feature **must** wreck the forecast. If WQL barely moves, the ablation
is silently doing nothing -- wrong module path, a copy instead of a view, grad still on --
and every number after this is noise dressed up as a finding.

In [ ]:
@torch.no_grad()
def ablate_layers(layer_idx, ins, tgt):
    saved = [(FFN[i], FFN[i].wi.weight.clone(), FFN[i].wo.weight.clone()) for i in layer_idx]
    for i in layer_idx:
        FFN[i].wi.weight.zero_(); FFN[i].wo.weight.zero_()
    s = score(ins, tgt)
    for mod, wi, wo in saved:
        mod.wi.weight.copy_(wi); mod.wo.weight.copy_(wo)
    return s

allz = ablate_layers(range(len(FFN)), ins, tgt)
print(f"baseline            WQL {base:8.4f}")
print(f"ALL FFN zeroed      WQL {allz:8.4f}   ({(allz-base)/base*100:+.0f}%)")
assert allz > base*1.5, "CONTROL FAILED - ablation is a no-op, stop and debug"
print("\ncontrol passed: the ablation genuinely perturbs the model\n")

print("each FFN layer alone:")
for i, nm in enumerate(NAMES):
    s = ablate_layers([i], ins, tgt)
    print(f"   {nm:<9} WQL {s:.4f}  ({(s-base)/base*100:+.1f}%)")

## 4 - Ablate every feature individually

Zero row $j$ of `wi` (what the feature reads) and column $j$ of `wo` (what it writes),
re-forecast, restore. ~100 ms each, so all 8192 takes ~14 min.

Set `N_PER_LAYER = 64` for a 90-second sampled version that shows the same picture.

In [ ]:
N_PER_LAYER = N_FEAT      # N_FEAT = exhaustive; try 64 for a quick pass

@torch.no_grad()
def ablate_features(picks, ins, tgt):
    saved = []
    for p in picks:
        L, j = int(p)//N_FEAT, int(p)%N_FEAT
        mod = FFN[L]
        saved.append((mod, j, mod.wi.weight[j].clone(), mod.wo.weight[:, j].clone()))
        mod.wi.weight[j] = 0; mod.wo.weight[:, j] = 0
    s = score(ins, tgt)
    for mod, j, row, col in saved:
        mod.wi.weight[j] = row; mod.wo.weight[:, j] = col
    return s

rng, rows, t0 = np.random.default_rng(0), [], time.time()
for L, nm in enumerate(NAMES):
    sel = range(N_FEAT) if N_PER_LAYER >= N_FEAT else rng.choice(N_FEAT, N_PER_LAYER, replace=False)
    for j in sel:
        rows.append({"layer": nm, "feature": int(j),
                     "delta": ablate_features([L*N_FEAT+int(j)], ins, tgt) - base})
    print(f"  {nm}: {len(rows)} done, {time.time()-t0:.0f}s", flush=True)

json.dump({"baseline": base, "rows": rows}, open("ablation_results.json","w"))
d = np.array([r["delta"] for r in rows])
print(f"\n{len(rows)} ablations in {time.time()-t0:.0f}s")
print(f"largest DAMAGE : {d.max():+.5f}  ({d.max()/base*100:+.2f}% of baseline)")
print(f"largest BENEFIT: {d.min():+.5f}  ({d.min()/base*100:+.2f}%)")
print(f"removal hurts {(d>0).mean()*100:.0f}% of features, helps {(d<0).mean()*100:.0f}%")
print("\ntop 5 most damaging:")
for r in sorted(rows, key=lambda r: -r["delta"])[:5]:
    print(f"   {r['layer']:>9} f{r['feature']:<5} {r['delta']:+.5f} ({r['delta']/base*100:+.2f}%)")
a = np.sort(np.abs(d))[::-1]
print(f"\nconcentration: top 10% of features hold {a[:len(a)//10].sum()/np.abs(d).sum()*100:.0f}% of total |effect|")

## 5 - How much of the FFN can you delete?

Single-feature ablation can look flat simply because effects are small. The sharper
question is how many features you can remove **together** before forecasting degrades.

In [ ]:
print(f"{'removed':>10}{'% of FFN':>10}{'WQL':>10}{'change':>10}")
curve = []
for k in (1, 10, 100, 500, 1000, 2048, 4096, 6144, TOTAL):
    if k == TOTAL:
        s = ablate_layers(range(len(FFN)), ins, tgt)
    else:
        s = np.mean([ablate_features(np.random.default_rng(sd).choice(TOTAL, k, replace=False), ins, tgt)
                     for sd in range(3)])
    curve.append((k, s))
    print(f"{k:>10}{k/TOTAL*100:>9.0f}%{s:>10.4f}{(s-base)/base*100:>+9.1f}%")

## 6 - Does it hold on other datasets?

One dataset proves nothing. ETTh1/h2 are hourly, ETTm1 is 15-minute -- same domain,
different sampling rate, and that turns out to matter.

In [ ]:
print(f"{'dataset':<10}{'baseline':>10}{'-25%':>9}{'-50%':>9}{'-75%':>9}{'-100%':>11}")
for name in ("ETTh1", "ETTm1", "ETTh2"):
    i2, t2 = make_windows(load_ett(name))
    b2 = score(i2, t2)
    fr = [np.mean([ablate_features(np.random.default_rng(sd).choice(TOTAL, int(TOTAL*f), replace=False), i2, t2)
                   for sd in range(3)]) for f in (0.25, 0.5, 0.75)]
    az = ablate_layers(range(len(FFN)), i2, t2)
    print(f"{name:<10}{b2:>10.4f}" + "".join(f"{(x-b2)/b2*100:>+8.1f}%" for x in fr)
          + f"{(az-b2)/b2*100:>+10.0f}%")

## What we found

On `chronos-bolt-tiny`, ETTh1, 56 windows:

| result | value |
|---|---|
| model vs naive last-value | 0.939 vs 2.055 -- genuinely forecasting |
| **all 8192 FFN features zeroed** | **+2312%** -- control passes |
| largest effect of any single feature | **+0.71%** |
| features that *improve* the forecast when removed | 28% |
| any single FFN layer removed | -1.4% to +1.6% |
| half the features removed | +0.3% (ETTh1), -2.4% (ETTh2), +10.9% (ETTm1) |

**The FFN is collectively essential and individually near-redundant.** No single feature
out of 8192 accounts for even 1% of forecasting accuracy, and more than a quarter of them
make the forecast slightly worse by existing. Effect is mildly concentrated -- the top 10%
of features hold ~67% of the total absolute effect, and the last encoder layer carries
2-4x the mean -- but the absolute scale stays sub-percent throughout.

This is the same distributed-storage picture the Titans ablation finds in a test-time
memory, now on a frozen forecaster judged by forecasting error. Storage being spread
across units is not peculiar to memory that updates at inference.

**What this does not show.** One model family, the smallest checkpoint (9M), three datasets
from a single domain. Redundancy is clearly dataset-dependent -- ETTm1 is roughly 30x
stricter than ETTh1 at the 50% level. Random ablation is also the *easy* case: a targeted
pruning method would likely remove more, and a reviewer may reasonably ask for
`chronos-bolt-base` and a broader benchmark such as GIFT-Eval.